In [11]:
import torch
import torch.nn.functional as F
import numpy as np
import evaluate
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

tokenizer = AutoTokenizer.from_pretrained("t5-base")
dataset = load_dataset("cnn_dailymail", "3.0.0")

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(text_target=examples["highlights"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"],  # fixed — matches T5's actual module names
    # remove layers_to_transform for now; verify it works first, then add back
)
model = get_peft_model(model, peft_config)

class ConsistencyTrainer(Seq2SeqTrainer):
    def __init__(self, *args, alpha=0.01, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs, output_attentions=True)
        task_loss = outputs.loss

        # guard: cross_attentions can be None during eval generate
        cross_attentions = outputs.cross_attentions
        consistency_loss = torch.tensor(0.0, device=task_loss.device)

        if cross_attentions is not None and len(cross_attentions) > 1 and model.training:
            num_layers = len(cross_attentions)
            for i in range(1, num_layers):
                prev_layer = cross_attentions[i - 1]  # (batch, heads, tgt, src)
                curr_layer = cross_attentions[i]
                prev_log = torch.log(prev_layer + 1e-9)
                curr_log = torch.log(curr_layer + 1e-9)
                kl_fwd = F.kl_div(curr_log, prev_layer, reduction='none').mean()
                kl_bwd = F.kl_div(prev_log, curr_layer, reduction='none').mean()
                consistency_loss += 0.5 * (kl_fwd + kl_bwd)
            consistency_loss = consistency_loss / (num_layers - 1)

        total_loss = task_loss + self.alpha * consistency_loss
        return (total_loss, outputs) if return_outputs else total_loss

rouge = evaluate.load("rouge")

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [12]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # NEW FIX: handle tuple output
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # If logits -> convert to token ids
    if len(predictions.shape) == 3:
        predictions = np.argmax(predictions, axis=-1)

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        k: round(v, 4)
        for k, v in result.items()
    }



In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-ot-lora-results",

    # evaluate frequently
    eval_strategy="steps",
    eval_steps=50,

    # save checkpoints frequently
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,

    # log training loss frequently
    logging_strategy="steps",
    logging_steps=50,

    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=3,

    predict_with_generate=True,
    fp16=True,
    # half_precision_backend="auto",

    # automatically load best checkpoint at the end
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,

    # optional but useful
    report_to="none"
)

trainer = ConsistencyTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(2000)),
    eval_dataset=tokenized_datasets["validation"].select(range(200)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    alpha=0.01
)

trainer.train()

# evaluate best checkpoint on test set
test_results = trainer.evaluate(
    eval_dataset=tokenized_datasets["test"].select(range(500))
)

print(test_results)

/Users/mariiaonyshchuk/Documents/📚 Study/MMML/Labs/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


In [ ]:
model.print_trainable_parameters()


trainable params: 294,912 || all params: 223,198,464 || trainable%: 0.1321
